In [4]:
import re
import math
import random
from pathlib import Path
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

sherlock_text = Path("data/sherlock.txt").read_text(encoding="utf-8")
pride_text = Path("data/pride.txt").read_text(encoding="utf-8")

all_sents = to_sentences(sherlock_text) + to_sentences(pride_text)
random.Random(42).shuffle(all_sents)

split_idx = int(0.85 * len(all_sents))
train_sents = all_sents[:split_idx]
test_sents = all_sents[split_idx:]
train_half = train_sents[: len(train_sents) // 2]

print(f"Total sentences : {len(all_sents):,}")
print(f"Train sentences : {len(train_sents):,} (Half slice: {len(train_half):,})")
print(f"Test sentences  : {len(test_sents):,}")

Total sentences : 14,648
Train sentences : 12,450 (Half slice: 6,225)
Test sentences  : 2,198


In [5]:
class NgramLM:
    def __init__(self, n=2, k=0.5, min_freq=2, lambdas=None):
        self.n = n
        self.k = k
        self.min_freq = min_freq
        self.lambdas = lambdas

    def _replace_unk(self, tokens):
        return [t if t in self.V else "<unk>" for t in tokens]

    def fit(self, sents):
        raw_counts = Counter(t for s in sents for t in to_tokens(s))
        self.V = {w for w, c in raw_counts.items() if c >= self.min_freq} | {"</s>", "<unk>"}
        self.tables = {}
        for order in range(1, self.n + 1):
            foll = defaultdict(Counter)
            for s in sents:
                toks = ["<s>"] * (order - 1) + self._replace_unk(to_tokens(s)) + ["</s>"]
                for i in range(len(toks) - order + 1):
                    ctx = tuple(toks[i : i + order - 1])
                    nxt = toks[i + order - 1]
                    foll[ctx][nxt] += 1
            self.tables[order] = foll
        return self

    def _order_prob(self, order, ctx, w):
        w = w if w in self.V else "<unk>"
        ctx = tuple(t if t in self.V or t == "<s>" else "<unk>" for t in ctx)
        c = self.tables[order].get(ctx, Counter())
        total = sum(c.values())
        return (c[w] + self.k) / (total + self.k * len(self.V))

    def prob(self, ctx, w):
        if not self.lambdas:
            return self._order_prob(self.n, ctx, w)
        p = 0.0
        for order, lam in enumerate(self.lambdas, start=1):
            sub_ctx = ctx[-(order - 1):] if order > 1 else ()
            p += lam * self._order_prob(order, sub_ctx, w)
        return p

    def suggest_next(self, text, topk=3):
        toks = self._replace_unk(to_tokens(text))
        for order in range(self.n, 0, -1):
            ctx = tuple((["<s>"] * (order - 1) + toks)[-(order - 1):]) if order > 1 else ()
            c = self.tables[order].get(ctx, Counter())
            words = [w for w, _ in c.most_common() if w not in ("</s>", "<unk>")]
            if words:
                return words[:topk]
        return ["(no idea)"]

    def generate(self, seed=0, max_len=30):
        rng = random.Random(seed)
        ctx = ("<s>",) * (self.n - 1)
        out = []
        for _ in range(max_len):
            c = None
            for order in range(self.n, 0, -1):
                sub_ctx = ctx[-(order - 1):] if order > 1 else ()
                cand = {w: cnt for w, cnt in self.tables[order].get(sub_ctx, {}).items() if w != "<unk>"}
                if cand:
                    c = cand
                    break
            if not c:
                break
            words, wts = zip(*c.items())
            nxt = rng.choices(words, weights=wts)[0]
            if nxt == "</s>":
                break
            out.append(nxt)
            if self.n > 1:
                ctx = (ctx + (nxt,))[-(self.n - 1):]
        return " ".join(out)

    def perplexity(self, sents):
        log_sum, N = 0.0, 0
        for s in sents:
            toks = ["<s>"] * (self.n - 1) + to_tokens(s) + ["</s>"]
            for i in range(len(toks) - self.n + 1):
                ctx = tuple(toks[i : i + self.n - 1])
                w = toks[i + self.n - 1]
                log_sum += math.log(self.prob(ctx, w))
                N += 1
        return math.exp(-log_sum / N)